# Environnement creation

In [2]:
%%bash
# Create the virtual environments
python -m venv env_qiskit_2_4_2
python -m venv env_qiskit_2_5_2

# Upgrade pip in both environments
./env_qiskit_2_4_2/bin/python -m pip install --upgrade pip -q
./env_qiskit_2_5_2/bin/python -m pip install --upgrade pip -q

# Install specific Qiskit versions
./env_qiskit_2_4_2/bin/python -m pip install qiskit==2.4.2 pandas qiskit_ibm_runtime numpy -q
./env_qiskit_2_5_2/bin/python -m pip install qiskit==2.5.2 pandas qiskit_ibm_runtime numpy -q

echo "Environments configured."

Environments configured.


In [3]:
%%script ./env_qiskit_2_4_2/bin/python
import qiskit

print(f"Running Qiskit: {qiskit.__version__}")

Running Qiskit: 2.4.2


In [4]:
%%script ./env_qiskit_2_5_2/bin/python
import qiskit

print(f"Running Qiskit: {qiskit.__version__}")

Running Qiskit: 2.5.2


# Sabre comparaison with transpile()

In this run, I investigated the two versions with the most used transpilation tool, which is transpile().

## Qiskit 2.4.2

In [5]:
%%script ./env_qiskit_2_4_2/bin/python

import os
import sys
import time
import glob
import pandas as pd
import qiskit
from qiskit import qasm2, transpile
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeNighthawk

def execute_transpile_benchmark(circuits_dir: str, output_csv: str = "transpile_benchmark.csv", number_seeds: int = 5, base_seed: int = 42):
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Target directory '{circuits_dir}' does not exist.")
        
    qasm_files = glob.glob(os.path.join(circuits_dir, "*.qasm"))
    if not qasm_files:
        sys.exit(f"FATAL: No .qasm files found in '{circuits_dir}'.")
        
    circuits = {}
    for f in qasm_files:
        file_name = os.path.basename(f)
        try:
            circuits[file_name] = qasm2.load(
                f,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Bypassing '{file_name}'. QASM parsing failed: {e}")
            continue

    if not circuits:
        sys.exit(f"FATAL: No valid QASM files could be parsed in '{circuits_dir}'. Pipeline terminated.")
        
    print(f"[+] Successfully loaded {len(circuits)} valid circuit(s) for benchmarking.")

    hardware_targets = {
        "Sherbrooke_127Q": FakeSherbrooke(),
        "Nighthawk_120Q": FakeNighthawk()
    }

    seeds = [base_seed + i for i in range(number_seeds)]
    records = []

    for backend_name, backend in hardware_targets.items():
        print(f"\n--- Initializing Target Architecture: {backend_name} ---")
        hw_qubits = backend.num_qubits 

        for circ_name, qc in circuits.items():
            logical_qubits = qc.num_qubits
            print(f"\nTarget Circuit: {circ_name} | Logical Qubits: {logical_qubits} | Hardware Limits: {hw_qubits}")
            
            if logical_qubits > hw_qubits:
                print(f"    [!] Violation: {circ_name} exceeds {backend_name} hardware limits. Bypassing.")
                continue

            for seed in seeds:
                try:
                    start_time = time.time()
                    
                    transpiled_qc = transpile(
                        qc,
                        backend=backend,
                        optimization_level=3,
                        seed_transpiler=seed
                    )
                    
                    exec_time = time.time() - start_time
                    
                    ops = transpiled_qc.count_ops()
                    total_physical_gates = sum(ops.values())
                    
                    count_2q = sum(
                        1 for inst in transpiled_qc.data 
                        if getattr(inst.operation, 'num_qubits', 0) >= 2 
                        and getattr(inst.operation, 'name', '') != 'barrier'
                    )

                    total_depth = transpiled_qc.depth()
                    
                    depth_2q = transpiled_qc.depth(
                        filter_function=lambda x: getattr(x.operation, 'num_qubits', 0) >= 2 
                                                  and getattr(x.operation, 'name', '') != 'barrier'
                    )

                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": round(exec_time, 4),
                        "Total Gates": total_physical_gates,
                        "2Q Gate Count": count_2q,
                        "Total Depth": total_depth,
                        "2Q Depth": depth_2q,
                        "Gate Breakdown": str(dict(ops)),
                        "Error": None
                    })
                    print(f"    [+] Seed {seed} compiled successfully in {exec_time:.2f}s. (2Q Gates: {count_2q}, Depth: {total_depth})")
                    
                except Exception as e:
                    print(f"    [!] Failure during transpile on seed {seed}: {str(e)}")
                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": None,
                        "Total Gates": None,
                        "2Q Gate Count": None,
                        "Total Depth": None,
                        "2Q Depth": None,
                        "Gate Breakdown": None,
                        "Error": str(e)
                    })

    if not records:
        sys.exit("FATAL: No successful transpilation records generated. Exiting.")

    df_raw = pd.DataFrame(records)
    
    df_success = df_raw[df_raw["Error"].isnull()].copy()
    
    numeric_cols = ["Execution Time (s)", "Total Gates", "2Q Gate Count", "Total Depth", "2Q Depth"]
    for col in numeric_cols:
        df_success.loc[:, col] = pd.to_numeric(df_success[col])

    group_keys = ["Backend", "Circuit"]
    
    df_avg = df_success.groupby(group_keys)[numeric_cols].mean().round(2).reset_index()
    
    success_counts = df_success.groupby(group_keys).size().reset_index(name="Successful_Runs")
    df_avg = pd.merge(df_avg, success_counts, on=group_keys, how="left")
    df_avg["Total_Seeds"] = number_seeds

    final_columns = group_keys + ["Successful_Runs", "Total_Seeds"] + numeric_cols
    df_avg = df_avg[final_columns]

    df_avg.to_csv(output_csv, index=False)
    
    print(f"{output_csv} generated.")
    
    return df_avg

if __name__ == "__main__":
    print(f"Running with Qiskit: {qiskit.__version__}")
    execute_transpile_benchmark(circuits_dir="circuits", output_csv="fully_transpiled_benchmark_2_4_2.csv", number_seeds=100, base_seed=420)

Running with Qiskit: 2.4.2
[+] Successfully loaded 88 valid circuit(s) for benchmarking.


/home/axel/Desktop/sabre_benchmark/env_qiskit_2_4_2/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



--- Initializing Target Architecture: Sherbrooke_127Q ---

Target Circuit: ising_n42.qasm | Logical Qubits: 42 | Hardware Limits: 127
    [+] Seed 420 compiled successfully in 0.07s. (2Q Gates: 82, Depth: 20)
    [+] Seed 421 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 422 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 423 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 424 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 425 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 426 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 427 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 428 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 429 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 430 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 20)
    [+] Seed 431 compiled successfully i

## Qiskit 2.5.2

In [6]:
%%script ./env_qiskit_2_5_2/bin/python

import os
import sys
import time
import glob
import pandas as pd
import qiskit
from qiskit import qasm2, transpile
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeNighthawk

def execute_transpile_benchmark(circuits_dir: str, output_csv: str = "transpile_benchmark.csv", number_seeds: int = 5, base_seed: int = 42):
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Target directory '{circuits_dir}' does not exist.")
        
    qasm_files = glob.glob(os.path.join(circuits_dir, "*.qasm"))
    if not qasm_files:
        sys.exit(f"FATAL: No .qasm files found in '{circuits_dir}'.")
        
    circuits = {}
    for f in qasm_files:
        file_name = os.path.basename(f)
        try:
            circuits[file_name] = qasm2.load(
                f,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Bypassing '{file_name}'. QASM parsing failed: {e}")
            continue

    if not circuits:
        sys.exit(f"FATAL: No valid QASM files could be parsed in '{circuits_dir}'. Pipeline terminated.")
        
    print(f"[+] Successfully loaded {len(circuits)} valid circuit(s) for benchmarking.")

    hardware_targets = {
        "Sherbrooke_127Q": FakeSherbrooke(),
        "Nighthawk_120Q": FakeNighthawk()
    }

    seeds = [base_seed + i for i in range(number_seeds)]
    records = []

    for backend_name, backend in hardware_targets.items():
        print(f"\n--- Initializing Target Architecture: {backend_name} ---")
        hw_qubits = backend.num_qubits 

        for circ_name, qc in circuits.items():
            logical_qubits = qc.num_qubits
            print(f"\nTarget Circuit: {circ_name} | Logical Qubits: {logical_qubits} | Hardware Limits: {hw_qubits}")
            
            if logical_qubits > hw_qubits:
                print(f"    [!] Violation: {circ_name} exceeds {backend_name} hardware limits. Bypassing.")
                continue

            for seed in seeds:
                try:
                    start_time = time.time()
                    
                    transpiled_qc = transpile(
                        qc,
                        backend=backend,
                        optimization_level=3,
                        seed_transpiler=seed
                    )
                    
                    exec_time = time.time() - start_time
                    
                    ops = transpiled_qc.count_ops()
                    total_physical_gates = sum(ops.values())
                    
                    count_2q = sum(
                        1 for inst in transpiled_qc.data 
                        if getattr(inst.operation, 'num_qubits', 0) >= 2 
                        and getattr(inst.operation, 'name', '') != 'barrier'
                    )

                    total_depth = transpiled_qc.depth()
                    
                    depth_2q = transpiled_qc.depth(
                        filter_function=lambda x: getattr(x.operation, 'num_qubits', 0) >= 2 
                                                  and getattr(x.operation, 'name', '') != 'barrier'
                    )

                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": round(exec_time, 4),
                        "Total Gates": total_physical_gates,
                        "2Q Gate Count": count_2q,
                        "Total Depth": total_depth,
                        "2Q Depth": depth_2q,
                        "Gate Breakdown": str(dict(ops)),
                        "Error": None
                    })
                    print(f"    [+] Seed {seed} compiled successfully in {exec_time:.2f}s. (2Q Gates: {count_2q}, Depth: {total_depth})")
                    
                except Exception as e:
                    print(f"    [!] Failure during transpile on seed {seed}: {str(e)}")
                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": None,
                        "Total Gates": None,
                        "2Q Gate Count": None,
                        "Total Depth": None,
                        "2Q Depth": None,
                        "Gate Breakdown": None,
                        "Error": str(e)
                    })

    if not records:
        sys.exit("FATAL: No successful transpilation records generated. Exiting.")

    df_raw = pd.DataFrame(records)
    
    df_success = df_raw[df_raw["Error"].isnull()].copy()
    
    numeric_cols = ["Execution Time (s)", "Total Gates", "2Q Gate Count", "Total Depth", "2Q Depth"]
    for col in numeric_cols:
        df_success.loc[:, col] = pd.to_numeric(df_success[col])

    group_keys = ["Backend", "Circuit"]
    
    df_avg = df_success.groupby(group_keys)[numeric_cols].mean().round(2).reset_index()
    
    success_counts = df_success.groupby(group_keys).size().reset_index(name="Successful_Runs")
    df_avg = pd.merge(df_avg, success_counts, on=group_keys, how="left")
    df_avg["Total_Seeds"] = number_seeds

    final_columns = group_keys + ["Successful_Runs", "Total_Seeds"] + numeric_cols
    df_avg = df_avg[final_columns]

    df_avg.to_csv(output_csv, index=False)
    
    print(f"{output_csv} generated.")

    
    return df_avg

if __name__ == "__main__":
    print(f"Running with Qiskit: {qiskit.__version__}")
    execute_transpile_benchmark(circuits_dir="circuits", output_csv="fully_transpiled_benchmark_2_5_2.csv", number_seeds=100, base_seed=420)

Running with Qiskit: 2.5.2
[+] Successfully loaded 88 valid circuit(s) for benchmarking.


/home/axel/Desktop/sabre_benchmark/env_qiskit_2_5_2/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



--- Initializing Target Architecture: Sherbrooke_127Q ---

Target Circuit: ising_n42.qasm | Logical Qubits: 42 | Hardware Limits: 127
    [+] Seed 420 compiled successfully in 0.11s. (2Q Gates: 82, Depth: 30)
    [+] Seed 421 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 422 compiled successfully in 0.06s. (2Q Gates: 82, Depth: 30)
    [+] Seed 423 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 424 compiled successfully in 0.06s. (2Q Gates: 82, Depth: 30)
    [+] Seed 425 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 426 compiled successfully in 0.07s. (2Q Gates: 82, Depth: 30)
    [+] Seed 427 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 428 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 429 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 30)
    [+] Seed 430 compiled successfully in 0.07s. (2Q Gates: 82, Depth: 30)
    [+] Seed 431 compiled successfully i

## Merging CSV

In [7]:
%%script ./env_qiskit_2_5_2/bin/python


import os
import sys
import numpy as np
import pandas as pd

def generate_comparison_matrix(csv_2_4_2: str, csv_2_5_2: str, output_csv: str):
    """
    Executes a statistical comparison between Qiskit 2.4.2 and 2.5.2 transpilation datasets.
    Dynamically maps 2Q metric columns to prevent schema mismatches between generators.
    """
    for file_path in [csv_2_4_2, csv_2_5_2]:
        if not os.path.isfile(file_path):
            sys.exit(f"FATAL: Required dataset '{file_path}' does not exist. Execution halted.")

    try:
        df1 = pd.read_csv(csv_2_4_2)
        df2 = pd.read_csv(csv_2_5_2)
    except Exception as e:
        sys.exit(f"FATAL: Data ingestion failed. Verify CSV integrity. Error: {e}")

    keys = ["Backend", "Circuit"]
    
    def resolve_2q_col(df):
        if "2Q Gate Count" in df.columns:
            return "2Q Gate Count"
        elif "2Q Count" in df.columns:
            return "2Q Count"
        return None
    
    col_2q_1 = resolve_2q_col(df1)
    col_2q_2 = resolve_2q_col(df2)
    
    if not col_2q_1 or not col_2q_2:
        sys.exit(f"FATAL: Could not locate a valid 2Q metric column ('2Q Gate Count' or '2Q Count') in one or both datasets.")
        
    if col_2q_1 != col_2q_2:
         df2 = df2.rename(columns={col_2q_2: col_2q_1})
         
    metrics = ["Execution Time (s)", "Total Gates", col_2q_1, "Total Depth", "2Q Depth", "Successful_Runs"]

    for df, name in zip([df1, df2], [csv_2_4_2, csv_2_5_2]):
        missing = [col for col in keys + metrics if col not in df.columns]
        if missing:
            sys.exit(f"FATAL: Dataset '{name}' is missing required columns: {missing}")

    print(f"[*] Ingested {len(df1)} records from {csv_2_4_2}")
    print(f"[*] Ingested {len(df2)} records from {csv_2_5_2}")

    df_merged = pd.merge(df1, df2, on=keys, how="outer", suffixes=('_2.4.2', '_2.5.2'))

    for m in metrics:
        col_242 = f"{m}_2.4.2"
        col_252 = f"{m}_2.5.2"
        abs_diff_col = f"{m}_Abs_Delta"
        pct_diff_col = f"{m}_%_Change"

        df_merged[abs_diff_col] = df_merged[col_252] - df_merged[col_242]

        if m != "Successful_Runs":
            df_merged[pct_diff_col] = np.where(
                df_merged[col_242] == 0,
                np.nan,
                (df_merged[abs_diff_col] / df_merged[col_242]) * 100
            )

    final_cols = keys.copy()
    for m in metrics:
        final_cols.extend([f"{m}_2.4.2", f"{m}_2.5.2", f"{m}_Abs_Delta"])
        if m != "Successful_Runs":
            final_cols.append(f"{m}_%_Change")

    df_final = df_merged[final_cols].copy()

    float_cols = df_final.select_dtypes(include=['float64']).columns
    df_final[float_cols] = df_final[float_cols].round(2)

    try:
        df_final.to_csv(output_csv, index=False)
        print(f"[+] Dimensional comparison complete. Matrix written to: {output_csv}")
    except Exception as e:
        sys.exit(f"FATAL: Could not write output file '{output_csv}'. Error: {e}")

    regressions_depth = df_final[df_final["2Q Depth_%_Change"] > 0]
    improvements_depth = df_final[df_final["2Q Depth_%_Change"] < 0]
    
    regressions_2q = df_final[df_final[f"{col_2q_1}_%_Change"] > 0]
    improvements_2q = df_final[df_final[f"{col_2q_1}_%_Change"] < 0]
    
    print(f"\n--- Matrix Summary (2.4.2 vs 2.5.2) ---")
    print(f"Total Circuits Analyzed: {len(df_final)}")
    print(f"Circuits with 2Q Depth Regressions in 2.5.2 : {len(regressions_depth)}")
    print(f"Circuits with 2Q Depth Improvements in 2.5.2: {len(improvements_depth)}")
    print(f"Circuits with {col_2q_1} Regressions in 2.5.2 : {len(regressions_2q)}")
    print(f"Circuits with {col_2q_1} Improvements in 2.5.2: {len(improvements_2q)}")
    print(f"---------------------------------------\n")

if __name__ == "__main__":
    baseline_dataset = "fully_transpiled_benchmark_2_4_2.csv"
    target_dataset = "fully_transpiled_benchmark_2_5_2.csv"
    output_matrix = "qiskit_version_fully_transpiled_comparison.csv"
    
    generate_comparison_matrix(
        csv_2_4_2=baseline_dataset, 
        csv_2_5_2=target_dataset, 
        output_csv=output_matrix
    )

[*] Ingested 176 records from fully_transpiled_benchmark_2_4_2.csv
[*] Ingested 176 records from fully_transpiled_benchmark_2_5_2.csv
[+] Dimensional comparison complete. Matrix written to: qiskit_version_fully_transpiled_comparison.csv

--- Matrix Summary (2.4.2 vs 2.5.2) ---
Total Circuits Analyzed: 176
Circuits with 2Q Depth Regressions in 2.5.2 : 56
Circuits with 2Q Depth Improvements in 2.5.2: 43
Circuits with 2Q Gate Count Regressions in 2.5.2 : 56
Circuits with 2Q Gate Count Improvements in 2.5.2: 42
---------------------------------------



# Sabre comparaison with routing only

In this run, I investigated if the routing was the reason of that difference. I recorded the circuits swap and depth right after the routing. We can see that routing is not responsible for that difference.

## Qiskit 2.4.2

In [8]:
%%script ./env_qiskit_2_4_2/bin/python

import os
import sys
import time
import glob
import pandas as pd
import qiskit
from qiskit import qasm2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler import StagedPassManager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeNighthawk

def execute_transpile_benchmark(circuits_dir: str, output_csv: str = "transpile_benchmark.csv", number_seeds: int = 5, base_seed: int = 42):
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Target directory '{circuits_dir}' does not exist.")
        
    qasm_files = glob.glob(os.path.join(circuits_dir, "*.qasm"))
    if not qasm_files:
        sys.exit(f"FATAL: No .qasm files found in '{circuits_dir}'.")
        
    circuits = {}
    for f in qasm_files:
        file_name = os.path.basename(f)
        try:
            circuits[file_name] = qasm2.load(
                f,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Bypassing '{file_name}'. QASM parsing failed: {e}")
            continue

    if not circuits:
        sys.exit(f"FATAL: No valid QASM files could be parsed in '{circuits_dir}'. Pipeline terminated.")
        
    print(f"[+] Successfully loaded {len(circuits)} valid circuit(s) for benchmarking.")

    hardware_targets = {
        "Sherbrooke_127Q": FakeSherbrooke(),
        "Nighthawk_120Q": FakeNighthawk()
    }

    seeds = [base_seed + i for i in range(number_seeds)]
    records = []

    for backend_name, backend in hardware_targets.items():
        print(f"\n--- Initializing Target Architecture: {backend_name} ---")
        hw_qubits = backend.num_qubits 

        for circ_name, qc in circuits.items():
            logical_qubits = qc.num_qubits
            print(f"\nTarget Circuit: {circ_name} | Logical Qubits: {logical_qubits} | Hardware Limits: {hw_qubits}")
            
            if logical_qubits > hw_qubits:
                print(f"    [!] Violation: {circ_name} exceeds {backend_name} hardware limits. Bypassing.")
                continue

            for seed in seeds:
                try:
                    start_time = time.time()
                    
                    preset_pm = generate_preset_pass_manager(
                        optimization_level=3,
                        backend=backend,
                        seed_transpiler=seed
                    )

                    init_pm = StagedPassManager(stages=["init"], init=preset_pm.init)
                    qc_init = init_pm.run(qc)

                    route_pm = StagedPassManager(
                        stages=["layout", "routing"],
                        layout=preset_pm.layout,
                        routing=preset_pm.routing
                    )
                    transpiled_qc = route_pm.run(qc_init)
                    
                    exec_time = time.time() - start_time
                    
                    ops = transpiled_qc.count_ops()
                    total_physical_gates = sum(ops.values())
                    
                    count_2q = sum(
                        1 for inst in transpiled_qc.data 
                        if getattr(inst.operation, 'num_qubits', 0) >= 2 
                        and getattr(inst.operation, 'name', '') != 'barrier'
                    )

                    total_depth = transpiled_qc.depth()
                    
                    depth_2q = transpiled_qc.depth(
                        filter_function=lambda x: getattr(x.operation, 'num_qubits', 0) >= 2 
                                                  and getattr(x.operation, 'name', '') != 'barrier'
                    )

                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": round(exec_time, 4),
                        "Total Gates": total_physical_gates,
                        "2Q Gate Count": count_2q,
                        "Total Depth": total_depth,
                        "2Q Depth": depth_2q,
                        "Gate Breakdown": str(dict(ops)),
                        "Error": None
                    })
                    print(f"    [+] Seed {seed} compiled successfully in {exec_time:.2f}s. (2Q Gates: {count_2q}, Depth: {total_depth})")
                    
                except Exception as e:
                    print(f"    [!] Failure during transpile on seed {seed}: {str(e)}")
                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": None,
                        "Total Gates": None,
                        "2Q Gate Count": None,
                        "Total Depth": None,
                        "2Q Depth": None,
                        "Gate Breakdown": None,
                        "Error": str(e)
                    })

    if not records:
        sys.exit("FATAL: No successful transpilation records generated. Exiting.")

    df_raw = pd.DataFrame(records)
    
    df_success = df_raw[df_raw["Error"].isnull()].copy()
    
    numeric_cols = ["Execution Time (s)", "Total Gates", "2Q Gate Count", "Total Depth", "2Q Depth"]
    for col in numeric_cols:
        df_success.loc[:, col] = pd.to_numeric(df_success[col])

    group_keys = ["Backend", "Circuit"]
    
    df_avg = df_success.groupby(group_keys)[numeric_cols].mean().round(2).reset_index()
    
    success_counts = df_success.groupby(group_keys).size().reset_index(name="Successful_Runs")
    df_avg = pd.merge(df_avg, success_counts, on=group_keys, how="left")
    df_avg["Total_Seeds"] = number_seeds

    final_columns = group_keys + ["Successful_Runs", "Total_Seeds"] + numeric_cols
    df_avg = df_avg[final_columns]


    df_avg.to_csv(output_csv, index=False)
    
    print(f"{output_csv} generated.")
    
    return df_avg

if __name__ == "__main__":
    print(f"Running with Qiskit: {qiskit.__version__}")
    execute_transpile_benchmark(circuits_dir="circuits", output_csv="routing_only_benchmark_2_4_2.csv", number_seeds=100, base_seed=420)

Running with Qiskit: 2.4.2
[+] Successfully loaded 88 valid circuit(s) for benchmarking.


/home/axel/Desktop/sabre_benchmark/env_qiskit_2_4_2/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



--- Initializing Target Architecture: Sherbrooke_127Q ---

Target Circuit: ising_n42.qasm | Logical Qubits: 42 | Hardware Limits: 127
    [+] Seed 420 compiled successfully in 0.07s. (2Q Gates: 82, Depth: 10)
    [+] Seed 421 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 422 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 423 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 424 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 425 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 426 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 427 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 428 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 429 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 430 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 431 compiled successfully i

## Qiskit 2.5.2

In [9]:
%%script ./env_qiskit_2_5_2/bin/python

import os
import sys
import time
import glob
import pandas as pd
import qiskit
from qiskit import qasm2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler import StagedPassManager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeNighthawk

def execute_transpile_benchmark(circuits_dir: str, output_csv: str = "transpile_benchmark.csv", number_seeds: int = 5, base_seed: int = 42):
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Target directory '{circuits_dir}' does not exist.")
        
    qasm_files = glob.glob(os.path.join(circuits_dir, "*.qasm"))
    if not qasm_files:
        sys.exit(f"FATAL: No .qasm files found in '{circuits_dir}'.")
        
    circuits = {}
    for f in qasm_files:
        file_name = os.path.basename(f)
        try:
            circuits[file_name] = qasm2.load(
                f,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Bypassing '{file_name}'. QASM parsing failed: {e}")
            continue

    if not circuits:
        sys.exit(f"FATAL: No valid QASM files could be parsed in '{circuits_dir}'. Pipeline terminated.")
        
    print(f"[+] Successfully loaded {len(circuits)} valid circuit(s) for benchmarking.")

    hardware_targets = {
        "Sherbrooke_127Q": FakeSherbrooke(),
        "Nighthawk_120Q": FakeNighthawk()
    }

    seeds = [base_seed + i for i in range(number_seeds)]
    records = []

    for backend_name, backend in hardware_targets.items():
        print(f"\n--- Initializing Target Architecture: {backend_name} ---")
        hw_qubits = backend.num_qubits 

        for circ_name, qc in circuits.items():
            logical_qubits = qc.num_qubits
            print(f"\nTarget Circuit: {circ_name} | Logical Qubits: {logical_qubits} | Hardware Limits: {hw_qubits}")
            
            if logical_qubits > hw_qubits:
                print(f"    [!] Violation: {circ_name} exceeds {backend_name} hardware limits. Bypassing.")
                continue

            for seed in seeds:
                try:
                    start_time = time.time()
                    
                    preset_pm = generate_preset_pass_manager(
                        optimization_level=3,
                        backend=backend,
                        seed_transpiler=seed
                    )

                    init_pm = StagedPassManager(stages=["init"], init=preset_pm.init)
                    qc_init = init_pm.run(qc)

                    route_pm = StagedPassManager(
                        stages=["layout", "routing"],
                        layout=preset_pm.layout,
                        routing=preset_pm.routing
                    )
                    transpiled_qc = route_pm.run(qc_init)
                    
                    exec_time = time.time() - start_time
                    
                    ops = transpiled_qc.count_ops()
                    total_physical_gates = sum(ops.values())
                    
                    count_2q = sum(
                        1 for inst in transpiled_qc.data 
                        if getattr(inst.operation, 'num_qubits', 0) >= 2 
                        and getattr(inst.operation, 'name', '') != 'barrier'
                    )

                    total_depth = transpiled_qc.depth()
                    
                    depth_2q = transpiled_qc.depth(
                        filter_function=lambda x: getattr(x.operation, 'num_qubits', 0) >= 2 
                                                  and getattr(x.operation, 'name', '') != 'barrier'
                    )

                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": round(exec_time, 4),
                        "Total Gates": total_physical_gates,
                        "2Q Gate Count": count_2q,
                        "Total Depth": total_depth,
                        "2Q Depth": depth_2q,
                        "Gate Breakdown": str(dict(ops)),
                        "Error": None
                    })
                    print(f"    [+] Seed {seed} compiled successfully in {exec_time:.2f}s. (2Q Gates: {count_2q}, Depth: {total_depth})")
                    
                except Exception as e:
                    print(f"    [!] Failure during transpile on seed {seed}: {str(e)}")
                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": None,
                        "Total Gates": None,
                        "2Q Gate Count": None,
                        "Total Depth": None,
                        "2Q Depth": None,
                        "Gate Breakdown": None,
                        "Error": str(e)
                    })

    if not records:
        sys.exit("FATAL: No successful transpilation records generated. Exiting.")

    df_raw = pd.DataFrame(records)
    
    df_success = df_raw[df_raw["Error"].isnull()].copy()
    
    numeric_cols = ["Execution Time (s)", "Total Gates", "2Q Gate Count", "Total Depth", "2Q Depth"]
    for col in numeric_cols:
        df_success.loc[:, col] = pd.to_numeric(df_success[col])

    group_keys = ["Backend", "Circuit"]
    
    df_avg = df_success.groupby(group_keys)[numeric_cols].mean().round(2).reset_index()
    
    success_counts = df_success.groupby(group_keys).size().reset_index(name="Successful_Runs")
    df_avg = pd.merge(df_avg, success_counts, on=group_keys, how="left")
    df_avg["Total_Seeds"] = number_seeds

    final_columns = group_keys + ["Successful_Runs", "Total_Seeds"] + numeric_cols
    df_avg = df_avg[final_columns]


    df_avg.to_csv(output_csv, index=False)
    
    print(f"{output_csv} generated.")
    
    return df_avg

if __name__ == "__main__":
    print(f"Running with Qiskit: {qiskit.__version__}")
    execute_transpile_benchmark(circuits_dir="circuits", output_csv="routing_only_benchmark_2_5_2.csv", number_seeds=100, base_seed=420)

Running with Qiskit: 2.5.2
[+] Successfully loaded 88 valid circuit(s) for benchmarking.


/home/axel/Desktop/sabre_benchmark/env_qiskit_2_5_2/lib/python3.12/site-packages/qiskit_ibm_runtime/fake_provider/backends/nighthawk/fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



--- Initializing Target Architecture: Sherbrooke_127Q ---

Target Circuit: ising_n42.qasm | Logical Qubits: 42 | Hardware Limits: 127
    [+] Seed 420 compiled successfully in 0.06s. (2Q Gates: 82, Depth: 10)
    [+] Seed 421 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 422 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 423 compiled successfully in 0.04s. (2Q Gates: 82, Depth: 10)
    [+] Seed 424 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 425 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 426 compiled successfully in 0.04s. (2Q Gates: 82, Depth: 10)
    [+] Seed 427 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 428 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 429 compiled successfully in 0.04s. (2Q Gates: 82, Depth: 10)
    [+] Seed 430 compiled successfully in 0.05s. (2Q Gates: 82, Depth: 10)
    [+] Seed 431 compiled successfully i

## Merging CSV

In [10]:
%%script ./env_qiskit_2_5_2/bin/python


import os
import sys
import numpy as np
import pandas as pd

def generate_comparison_matrix(csv_2_4_2: str, csv_2_5_2: str, output_csv: str):
    """
    Executes a statistical comparison between Qiskit 2.4.2 and 2.5.2 transpilation datasets.
    Dynamically maps 2Q metric columns to prevent schema mismatches between generators.
    """
    for file_path in [csv_2_4_2, csv_2_5_2]:
        if not os.path.isfile(file_path):
            sys.exit(f"FATAL: Required dataset '{file_path}' does not exist. Execution halted.")

    try:
        df1 = pd.read_csv(csv_2_4_2)
        df2 = pd.read_csv(csv_2_5_2)
    except Exception as e:
        sys.exit(f"FATAL: Data ingestion failed. Verify CSV integrity. Error: {e}")

    keys = ["Backend", "Circuit"]
    
    def resolve_2q_col(df):
        if "2Q Gate Count" in df.columns:
            return "2Q Gate Count"
        elif "2Q Count" in df.columns:
            return "2Q Count"
        return None
    
    col_2q_1 = resolve_2q_col(df1)
    col_2q_2 = resolve_2q_col(df2)
    
    if not col_2q_1 or not col_2q_2:
        sys.exit(f"FATAL: Could not locate a valid 2Q metric column ('2Q Gate Count' or '2Q Count') in one or both datasets.")
        
    if col_2q_1 != col_2q_2:
         df2 = df2.rename(columns={col_2q_2: col_2q_1})
         
    metrics = ["Execution Time (s)", "Total Gates", col_2q_1, "Total Depth", "2Q Depth", "Successful_Runs"]

    for df, name in zip([df1, df2], [csv_2_4_2, csv_2_5_2]):
        missing = [col for col in keys + metrics if col not in df.columns]
        if missing:
            sys.exit(f"FATAL: Dataset '{name}' is missing required columns: {missing}")

    print(f"[*] Ingested {len(df1)} records from {csv_2_4_2}")
    print(f"[*] Ingested {len(df2)} records from {csv_2_5_2}")

    df_merged = pd.merge(df1, df2, on=keys, how="outer", suffixes=('_2.4.2', '_2.5.2'))

    for m in metrics:
        col_242 = f"{m}_2.4.2"
        col_252 = f"{m}_2.5.2"
        abs_diff_col = f"{m}_Abs_Delta"
        pct_diff_col = f"{m}_%_Change"

        df_merged[abs_diff_col] = df_merged[col_252] - df_merged[col_242]

        if m != "Successful_Runs":
            df_merged[pct_diff_col] = np.where(
                df_merged[col_242] == 0,
                np.nan,
                (df_merged[abs_diff_col] / df_merged[col_242]) * 100
            )

    final_cols = keys.copy()
    for m in metrics:
        final_cols.extend([f"{m}_2.4.2", f"{m}_2.5.2", f"{m}_Abs_Delta"])
        if m != "Successful_Runs":
            final_cols.append(f"{m}_%_Change")

    df_final = df_merged[final_cols].copy()

    float_cols = df_final.select_dtypes(include=['float64']).columns
    df_final[float_cols] = df_final[float_cols].round(2)

    try:
        df_final.to_csv(output_csv, index=False)
        print(f"[+] Dimensional comparison complete. Matrix written to: {output_csv}")
    except Exception as e:
        sys.exit(f"FATAL: Could not write output file '{output_csv}'. Error: {e}")

    regressions_depth = df_final[df_final["2Q Depth_%_Change"] > 0]
    improvements_depth = df_final[df_final["2Q Depth_%_Change"] < 0]
    
    regressions_2q = df_final[df_final[f"{col_2q_1}_%_Change"] > 0]
    improvements_2q = df_final[df_final[f"{col_2q_1}_%_Change"] < 0]
    
    print(f"\n--- Matrix Summary (2.4.2 vs 2.5.2) ---")
    print(f"Total Circuits Analyzed: {len(df_final)}")
    print(f"Circuits with 2Q Depth Regressions in 2.5.2 : {len(regressions_depth)}")
    print(f"Circuits with 2Q Depth Improvements in 2.5.2: {len(improvements_depth)}")
    print(f"Circuits with {col_2q_1} Regressions in 2.5.2 : {len(regressions_2q)}")
    print(f"Circuits with {col_2q_1} Improvements in 2.5.2: {len(improvements_2q)}")
    print(f"---------------------------------------\n")

if __name__ == "__main__":
    baseline_dataset = "routing_only_benchmark_2_4_2.csv"
    target_dataset = "routing_only_benchmark_2_5_2.csv"
    output_matrix = "qiskit_version_routing_only_comparison.csv"
    
    generate_comparison_matrix(
        csv_2_4_2=baseline_dataset, 
        csv_2_5_2=target_dataset, 
        output_csv=output_matrix
    )

[*] Ingested 176 records from routing_only_benchmark_2_4_2.csv
[*] Ingested 176 records from routing_only_benchmark_2_5_2.csv
[+] Dimensional comparison complete. Matrix written to: qiskit_version_routing_only_comparison.csv

--- Matrix Summary (2.4.2 vs 2.5.2) ---
Total Circuits Analyzed: 176
Circuits with 2Q Depth Regressions in 2.5.2 : 58
Circuits with 2Q Depth Improvements in 2.5.2: 36
Circuits with 2Q Gate Count Regressions in 2.5.2 : 63
Circuits with 2Q Gate Count Improvements in 2.5.2: 25
---------------------------------------

